# Food recognition training

This notebook shows how I fine-tuned ResNet18 on Food-101.

## Read before running
- The saved output is from the original Google Colab GPU run. It shows five completed epochs (training rounds) and an interruption during epoch six.
- The best recorded validation accuracy is 76.95% at epoch four.
- The official test split was used to check progress and select the best model. There is no separate final test result.
- Training downloads the dataset and ImageNet weights. Allow several GB of storage and use a GPU if possible.
- For local use, install requirements-training.txt. On Windows, use num_workers=0 in both data loaders if workers cause problems.
- Run the training cells in order. The export cell saves the current in-memory model, not necessarily the best model.
- The original export used a Google Drive path after the interrupted sixth epoch. The export code below now saves locally. Its old output is retained as historical evidence and has not been rerun.
- The demo checkpoint's name does not prove it contains exactly five completed epochs. Its accuracy has not been checked separately.
- Launch the working demo from the repository's app.py. This notebook does not overwrite that app.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from tqdm import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [5]:
train_data = datasets.Food101(root="data", split="train", download=True, transform=train_transform)
val_data   = datasets.Food101(root="data", split="test",  download=True, transform=val_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

print("Train samples:", len(train_data))
print("Val samples:", len(val_data))

100%|██████████| 5.00G/5.00G [04:05<00:00, 20.4MB/s]


Train samples: 75750
Val samples: 25250


In [6]:
num_classes = 101

model = models.resnet18(weights="IMAGENET1K_V1")

# replace final layer (fc)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [11]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.3
)

In [12]:
def train_one_epoch():
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        _, preds = torch.max(out, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / len(train_loader.dataset), correct / total


In [13]:
def validate():
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in tqdm(val_loader):
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            running_loss += loss.item() * x.size(0)
            _, preds = torch.max(out, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return running_loss / len(val_loader.dataset), correct / total


In [14]:
best_loss = float("inf")

for epoch in range(15):  # you can increase to 20–30 for better accuracy
    print(f"\nEPOCH {epoch+1}/15")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")

    # Save best model
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "resnet_food101.pth")
        print("🔥 Best model saved!")



EPOCH 1/15


100%|██████████| 790/790 [01:59<00:00,  6.59it/s]


Train Loss: 1.2881 | Train Acc: 66.67%
Val Loss:   1.0062 | Val Acc:   72.67%
🔥 Best model saved!

EPOCH 2/15


100%|██████████| 790/790 [02:03<00:00,  6.40it/s]


Train Loss: 1.0790 | Train Acc: 71.50%
Val Loss:   0.9354 | Val Acc:   74.27%
🔥 Best model saved!

EPOCH 3/15


...

Train Loss: 0.8402 | Train Acc: 77.24%
Val Loss:   0.8461 | Val Acc:   76.95%
🔥 Best model saved!

EPOCH 5/15


100%|██████████| 790/790 [01:51<00:00,  7.10it/s]


Train Loss: 0.7596 | Train Acc: 79.17%
Val Loss:   0.8480 | Val Acc:   76.91%

EPOCH 6/15


 36%|███▌      | 842/2368 [03:32<06:26,  3.95it/s]Process Process-26:
Process Process-25:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 219, in _finalize_join
    thread.join()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
       

KeyboardInterrupt: 

In [16]:
# Save the current model and matching labels in the notebook folder.
# This overwrites the demo checkpoint if you run it in the project folder.
# The current model may differ from the best model saved during training.
torch.save(model.state_dict(), "food101_after5epochs.pth")
import json
with open("food_classes.json", "w", encoding="utf-8") as file:
    json.dump(train_data.classes, file)
print("Model and labels saved successfully!")


Model saved successfully!


## Run the demo

The working app is in the repository's app.py. Run python -m streamlit run app.py from the project folder.

An old cell wrote an unrelated SimpleCNN app. It has been removed because that architecture does not match the trained ResNet18 model.


In [24]:
import torchvision.datasets as datasets
import json

food101 = datasets.Food101(root="./data", split="train", download=True)
class_list = food101.classes

with open("food_classes.json", "w") as f:
    json.dump(class_list, f)

print("food_classes.json created with", len(class_list), "classes")


food_classes.json created with 101 classes
